In [0]:
import requests
# https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGK&QO_fu146_anzr=b0-gvzr

In [0]:
# 1. Fetch Secrets from the Databricks Vault (dbutils is native to the notebook)
openweather_key = dbutils.secrets.get(scope="aviation_api_keys", key="openweather_key")
airlabs_key = dbutils.secrets.get(scope="aviation_api_keys", key="airlabs_key")

print("1. Secrets retrieved successfully!")

In [0]:
# 2. Test AirLabs Schedules API (Flights departing Little Rock - LIT)
print("2. Requesting AirLabs Flight Schedules...")
airlabs_url = "https://airlabs.co/api/v9/schedules"
airlabs_params = {
    "api_key": airlabs_key,
    "dep_iata": "LIT" # Little Rock Airport Code
}

airlabs_response = requests.get(airlabs_url, params=airlabs_params)

if airlabs_response.status_code == 200:
    flights = airlabs_response.json().get("response", [])
    print(f"   -> AirLabs Data Ping: SUCCESS. Found {len(flights)} scheduled flights departing LIT.")
    
    # Print the first flight to verify the delay and status columns exist
    if flights:
        first_flight = flights[0]
        flight_num = first_flight.get('flight_iata', 'Unknown')
        status = first_flight.get('status', 'Unknown')
        delay = first_flight.get('dep_delayed', 0)
        print(f"      Sample Data -> Flight: {flight_num} | Status: {status} | Delay: {delay} mins")
else:
    print(f"   -> AirLabs API Failed: {airlabs_response.status_code} - {airlabs_response.text}")

In [0]:
# 3. Test OpenWeather API (Current weather at LIT coordinates)
print("3. Requesting OpenWeather Data...")
weather_url = "https://api.openweathermap.org/data/2.5/weather"
weather_params = {
    "lat": 34.7294, # LIT Airport latitude
    "lon": -92.2242, # LIT Airport longitude
    "appid": openweather_key,
    "units": "imperial"
}

weather_response = requests.get(weather_url, params=weather_params)
if weather_response.status_code == 200:
    temp = weather_response.json().get("main", {}).get("temp")
    condition = weather_response.json().get("weather", [{}])[0].get("description")
    print(f"   -> OpenWeather API Ping: SUCCESS. It is currently {temp}°F with {condition} at LIT.")
else:
    print(f"   -> OpenWeather API Failed: {weather_response.status_code} - {weather_response.text}")

In [0]:
# 4. Initialize the Medallion Database Schema
spark.sql("CREATE SCHEMA IF NOT EXISTS aviation_project")
print("4. Medallion Schema 'aviation_project' verified successfully.")